<a href="https://colab.research.google.com/github/panaddatappoomee/food-selection-data-science/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ขั้นที่ 1

In [467]:
!pip install openpyxl
import pandas as pd

In [468]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 1. ติดตั้งฟอนต์ภาษาไทยลงในระบบ Colab
!apt-get -y install fonts-thai-tlwg > /dev/null 2>&1

# 2. โหลดไฟล์ฟอนต์เข้า Memory และตั้งค่า font_prop
font_path = "/usr/share/fonts/truetype/tlwg/Laksaman.ttf"
font_prop = fm.FontProperties(fname=font_path)

# 3. ตั้งค่า Default ให้ Matplotlib รู้จักฟอนต์และรองรับเครื่องหมายลบ
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

# ขั้นที่ 2 — ออกแบบ Class

In [494]:
import random
from datetime import datetime, timedelta

In [495]:
class Book:
    def __init__(self, book_id, title, author, category_code, category_name, status="ว่าง"):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.category_code = category_code
        self.category_name = category_name
        self.is_available = (status == "ว่าง")

    def borrow(self):
        """เปลี่ยนสถานะเป็นถูกยืม"""
        self.is_available = False

    def return_book(self):
        """เปลี่ยนสถานะกลับเป็นว่าง"""
        self.is_available = True


class Member:
    def __init__(self, member_id, name, member_type, loan_period_days):
        self.member_id = member_id
        self.name = name
        self.member_type = member_type
        self.loan_period_days = loan_period_days


class BookLoan:
    FINE_PER_DAY = 5

    def __init__(self, loan_id, book, member, borrow_date):
        self.loan_id = loan_id
        self.book = book
        self.member = member
        self.borrow_date = borrow_date

        # กำหนดวันส่งคืนตามสิทธิ์สมาชิก
        self.due_date = borrow_date + timedelta(days=member.loan_period_days)
        self.return_date = None

        # บันทึกว่าหนังสือถูกยืมทันทีเมื่อสร้างรายการ
        self.book.borrow()

    def calculate_late_fee(self):
        """คำนวณค่าปรับถ้าคืนช้ากว่ากำหนด"""
        if self.return_date is not None and self.return_date > self.due_date:
            late_days = (self.return_date - self.due_date).days
            return late_days * self.FINE_PER_DAY

        return 0

    def mark_returned(self, return_date):
        """บันทึกว่าคืนหนังสือแล้ว"""

        self.return_date = return_date
        self.book.return_book()

# ขั้นที่ 3 — เขียนฟังก์ชันช่วยงาน (Helper Functions)

In [496]:
import pandas as pd


def load_books(csv_path):
    """โหลดหนังสือจากไฟล์ CSV แล้วสร้างเป็น Book object"""

    df = pd.read_csv("/content/books.csv")

    return [
        Book(
            row.book_id,
            row.title,
            row.author,
            row.category_code,
            row.category_name,
            row.status
        )
        for row in df.itertuples(index=False)
    ]


def load_members(csv_path):
    """โหลดสมาชิกจากไฟล์ CSV แล้วสร้างเป็น Member object"""

    df = pd.read_csv("/content/members.csv")

    return [
        Member(
            row.member_id,
            row.name,
            row.member_type,
            row.loan_period_days
        )
        for row in df.itertuples(index=False)
    ]


def random_borrow_date(start_date, days_range=300):
    """ฟังก์ชัน: สุ่มวันที่เริ่มยืมหนังสือจากวันที่กำหนด -> คืนค่าเป็น datetime"""

    offset = random.randint(0, days_range)

    return start_date + timedelta(days=offset)


def decide_return_outcome(
    never_return_probability=0.20,
    late_probability=0.25
):
    """ฟังก์ชัน: สุ่มสถานะผลลัพธ์การคืนหนังสือตามความน่าจะเป็น -> คืนค่าเป็น string"""

    r = random.random()

    if r < never_return_probability:
        return "not_returned"

    elif r < never_return_probability + late_probability:
        return "late"

    else:
        return "on_time"


def calculate_return_date(due_date, outcome):
    """ฟังก์ชัน: คำนวณวันที่นำหนังสือมาคืนจริงตามสถานะผลลัพธ์ -> คืนค่าเป็น datetime หรือ None"""

    if outcome == "not_returned":
        return None

    elif outcome == "late":
        return due_date + timedelta(days=random.randint(1, 10))

    else:
        return due_date - timedelta(days=random.randint(0, 5))

# ทดสอบฟังก์ชันทีละตัว ก่อนเอาไปประกอบเป็นกระบวนการ

In [502]:
print("ทดสอบที่ 1: load_books และ load_members จากไฟล์จริง")
try:
    books = load_books("/content/books.csv")
    members = load_members("/content/members.csv")

    print(f"✅ โหลดหนังสือสำเร็จทั้งหมด: {len(books)} เล่ม")
    print(
        f"   ตัวอย่างเล่มแรก: {books[0].title} (สถานะพร้อมยืม: {books[0].is_available})"
    )

    print(f"✅ โหลดสมาชิกสำเร็จทั้งหมด: {len(members)} คน")
    print(
        f"   ตัวอย่างคนแรก: {members[0].name} (ยืมได้ {members[0].loan_period_days} วัน)"
    )
except FileNotFoundError as e:
    print(
        f"❌ ไม่พบไฟล์ CSV! กรุณาเช็กชื่อและพาธไฟล์อีกครั้ง: {e.filename}"
    )
except Exception as e:
    print(f"❌ เกิดข้อผิดพลาดในการอ่านไฟล์: {e}")

print("-" * 50)

ทดสอบที่ 1: load_books และ load_members จากไฟล์จริง
✅ โหลดหนังสือสำเร็จทั้งหมด: 150 เล่ม
   ตัวอย่างเล่มแรก: พื้นฐานการเขียนโปรแกรมด้วย Python (สถานะพร้อมยืม: True)
✅ โหลดสมาชิกสำเร็จทั้งหมด: 350 คน
   ตัวอย่างคนแรก: รติ (ยืมได้ 14 วัน)
--------------------------------------------------


In [504]:
print("ทดสอบที่ 2: random_borrow_date")
start_date = datetime(2026, 1, 1)
sample_borrow_date = random_borrow_date(start_date, days_range=30)
print(f"วันเริ่มสุ่ม: {start_date.strftime('%Y-%m-%d')}")
print(f"วันที่สุ่มได้: {sample_borrow_date.strftime('%Y-%m-%d')}")
print("-" * 50)

ทดสอบที่ 2: random_borrow_date
วันเริ่มสุ่ม: 2026-01-01
วันที่สุ่มได้: 2026-01-20
--------------------------------------------------


In [505]:
print("ทดสอบที่ 3: decide_return_outcome")
results = [decide_return_outcome() for _ in range(5)]
print(f"ผลการสุ่มคืนหนังสือ 5 ครั้ง: {results}")
print("-" * 50)

ทดสอบที่ 3: decide_return_outcome
ผลการสุ่มคืนหนังสือ 5 ครั้ง: ['on_time', 'on_time', 'late', 'on_time', 'late']
--------------------------------------------------


In [506]:
print("ทดสอบที่ 4: calculate_return_date")
due_date = datetime(2026, 8, 10)

return_on_time = calculate_return_date(due_date, "on_time")
return_late = calculate_return_date(due_date, "late")
return_never = calculate_return_date(due_date, "not_returned")

print(f"กำหนดคืน: {due_date.strftime('%Y-%m-%d')}")
print(
    f"กรณี คืนตรงเวลา (on_time): {return_on_time.strftime('%Y-%m-%d') if return_on_time else 'ไม่คืน'}"
)
print(
    f"กรณี คืนสาย (late): {return_late.strftime('%Y-%m-%d') if return_late else 'ไม่คืน'}"
)
print(f"กรณี ไม่คืนเลย (not_returned): {return_never}")
print("-" * 50)

ทดสอบที่ 4: calculate_return_date
กำหนดคืน: 2026-08-10
กรณี คืนตรงเวลา (on_time): 2026-08-05
กรณี คืนสาย (late): 2026-08-12
กรณี ไม่คืนเลย (not_returned): None
--------------------------------------------------


In [482]:
import random
from datetime import datetime, timedelta


def format_currency(amount, symbol="บาท"):
    """จัดรูปแบบตัวเลขเป็นสตริงเงิน เช่น 15.00 บาท"""
    return f"{amount:,.2f} {symbol}"


def simulate_member_visit(loan_id, member, book):
    """รับข้อมูลรหัสรายการ, สมาชิก, และหนังสือ -> จำลองกระบวนการยืมและคืน"""
    # กำหนดวันที่ยืม
    borrow_date = datetime(2025, 1, 1) + timedelta(days=random.randint(0, 30))

    # สร้างรายการยืม
    loan = BookLoan(loan_id, book, member, borrow_date)

    # ประมวลผลการคืนหนังสือ
    outcome = decide_return_outcome()
    return_date = calculate_return_date(loan.due_date, outcome)

    if return_date is not None:
        loan.mark_returned(return_date)

    # แสดงผลลัพธ์แบบเล่าเรื่อง
    print("=" * 60)
    print(
        f"📖 ออเดอร์ #{loan.loan_id}: คุณ '{member.name}' (รหัสสมาชิก:"
        f" {member.member_id}) เข้ามายืมหนังสือ"
    )
    print(
        f"📚 หนังสือที่เลือก: '{book.title}' (รหัสหนังสือ: {book.book_id}) โดย"
        f" {book.author}"
    )
    print(f"🗓️ วันที่ยืม: {borrow_date.strftime('%Y-%m-%d')}")
    print(f"⏰ กำหนดคืน: {loan.due_date.strftime('%Y-%m-%d')}")

    if loan.return_date:
        late_fee = loan.calculate_late_fee()
        print(
            f"✅ คืนหนังสือแล้วในวันที่ {loan.return_date.strftime('%Y-%m-%d')}"
        )
        if late_fee > 0:
            print(
                f"⚠️ เกินกำหนดส่งคืน! ค่าปรับสุทธิ: {format_currency(late_fee)}"
            )
        else:
            print("🎉 คืนตรงเวลา ไม่มีค่าปรับ")
    else:
        print("⚠️ ยังไม่คืนหนังสือ")
    print("=" * 60)

    return loan

In [483]:
# --- เรียกใช้งานจริงกับสมาชิก 1 คน ---

# 1. โหลดข้อมูลจาก CSV
members = load_members("members.csv")
books = load_books("books.csv")

# 2. เลือกสมาชิกคนแรก และสุ่มหนังสือ 1 เล่มจาก CSV
member_a = members[0]
book_a = random.choice(books)

# 3. เรียกใช้งานฟังก์ชันจำลองสำหรับสมาชิกคนแรก (ออเดอร์ #1)
order_a = simulate_member_visit(loan_id="L001", member=member_a, book=book_a)

📖 ออเดอร์ #L001: คุณ 'รติ' (รหัสสมาชิก: PS0001) เข้ามายืมหนังสือ
📚 หนังสือที่เลือก: 'พิธีกรรมและประเพณีทางศาสนา' (รหัสหนังสือ: D200-009) โดย จรัส วัฒนธรรม
🗓️ วันที่ยืม: 2025-01-20
⏰ กำหนดคืน: 2025-02-03
✅ คืนหนังสือแล้วในวันที่ 2025-02-01
🎉 คืนตรงเวลา ไม่มีค่าปรับ


In [484]:
members = load_members("members.csv")
books = load_books("books.csv")

# สุ่มเลือกสมาชิกที่จะเดินเข้ามาในวันนี้จำนวน 5 คน
walk_in_members = random.sample(members, k=min(5, len(members)))

completed_orders = []  # เก็บผลลัพธ์ของทุกออเดอร์ที่จำลองในรอบนี้

# เรียก simulate_member_visit() ซ้ำหลายๆ ครั้งผ่าน loop
for i, mem in enumerate(walk_in_members, start=1):
    loan_code = f"L{i:03d}"  # สร้างรหัสเช่น L001, L002, L003...
    selected_book = random.choice(books)  # สุ่มหนังสือที่สมาชิกเลือก

    # เรียกใช้ฟังก์ชันจำลอง
    order = simulate_member_visit(
        loan_id=loan_code, member=mem, book=selected_book
    )
    completed_orders.append(order)

print("=" * 60)
print(
    f"🏁 จบรอบสาธิต — วันนี้มีสมาชิกเข้ามาใช้บริการห้องสมุดทั้งหมด"
    f" {len(completed_orders)} คน"
)

📖 ออเดอร์ #L001: คุณ 'โอโซน' (รหัสสมาชิก: PS0034) เข้ามายืมหนังสือ
📚 หนังสือที่เลือก: 'การเขียนบทความสร้างสรรค์' (รหัสหนังสือ: D800-014) โดย นักเขียนอิสระ ความคิดใหม่
🗓️ วันที่ยืม: 2025-01-26
⏰ กำหนดคืน: 2025-02-09
⚠️ ยังไม่คืนหนังสือ
📖 ออเดอร์ #L002: คุณ 'ภา' (รหัสสมาชิก: PT0011) เข้ามายืมหนังสือ
📚 หนังสือที่เลือก: 'อาณาจักรโบราณในดินแดนไทย' (รหัสหนังสือ: D900-014) โดย นักประวัติศาสตร์ สุโขทัยอยุธยา
🗓️ วันที่ยืม: 2025-01-04
⏰ กำหนดคืน: 2025-02-03
⚠️ ยังไม่คืนหนังสือ
📖 ออเดอร์ #L003: คุณ 'ชะเอม' (รหัสสมาชิก: PS0185) เข้ามายืมหนังสือ
📚 หนังสือที่เลือก: 'ประวัติศาสตร์ไทยเบื้องต้น' (รหัสหนังสือ: D900-001) โดย นักประวัติศาสตร์ อดีตกาล
🗓️ วันที่ยืม: 2025-01-01
⏰ กำหนดคืน: 2025-01-15
⚠️ ยังไม่คืนหนังสือ
📖 ออเดอร์ #L004: คุณ 'มัชฌิ์' (รหัสสมาชิก: PS0213) เข้ามายืมหนังสือ
📚 หนังสือที่เลือก: 'เคมีอินทรีย์เบื้องต้น' (รหัสหนังสือ: D500-011) โดย คาร์บอน โมเลกุล
🗓️ วันที่ยืม: 2025-01-11
⏰ กำหนดคืน: 2025-01-25
✅ คืนหนังสือแล้วในวันที่ 2025-01-21
🎉 คืนตรงเวลา ไม่มีค่าปรับ
📖 ออเดอร์ #L005: คุณ 'นุ่น' 

In [485]:
# รวมออเดอร์แรก (L001) เข้ากับออเดอร์ที่เหลือจากในลูป (L002, L003, L004)
all_demo_orders = [order_a] + completed_orders

print("=" * 60)
print(f"📊 สรุปภาพรวมการจำลองทั้งหมด:")
print(f"   จำนวนออเดอร์ทั้งหมด: {len(all_demo_orders)} รายการ")
for order in all_demo_orders:
    fee = order.calculate_late_fee()
    fee_str = format_currency(fee) if fee > 0 else "ไม่มีค่าปรับ"
    print(f"   • รหัส: {order.loan_id} | สมาชิก: {order.member.name} | หนังสือ: {order.book.title} | ค่าปรับ: {fee_str}")
print("=" * 60)

📊 สรุปภาพรวมการจำลองทั้งหมด:
   จำนวนออเดอร์ทั้งหมด: 6 รายการ
   • รหัส: L001 | สมาชิก: รติ | หนังสือ: พิธีกรรมและประเพณีทางศาสนา | ค่าปรับ: ไม่มีค่าปรับ
   • รหัส: L001 | สมาชิก: โอโซน | หนังสือ: การเขียนบทความสร้างสรรค์ | ค่าปรับ: ไม่มีค่าปรับ
   • รหัส: L002 | สมาชิก: ภา | หนังสือ: อาณาจักรโบราณในดินแดนไทย | ค่าปรับ: ไม่มีค่าปรับ
   • รหัส: L003 | สมาชิก: ชะเอม | หนังสือ: ประวัติศาสตร์ไทยเบื้องต้น | ค่าปรับ: ไม่มีค่าปรับ
   • รหัส: L004 | สมาชิก: มัชฌิ์ | หนังสือ: เคมีอินทรีย์เบื้องต้น | ค่าปรับ: ไม่มีค่าปรับ
   • รหัส: L005 | สมาชิก: นุ่น | หนังสือ: การพัฒนาเว็บไซต์สมัยใหม่ | ค่าปรับ: 10.00 บาท


In [487]:
import pandas as pd

summary_rows = [
    {
        "loan_id": o.loan_id,
        "member_id": o.member.member_id,
        "member_name": o.member.name,
        "book_title": o.book.title,
        "borrow_date": o.borrow_date.strftime("%Y-%m-%d"),
        "due_date": o.due_date.strftime("%Y-%m-%d"),
        "return_date": (
            o.return_date.strftime("%Y-%m-%d") if o.return_date else "ยังไม่คืน"
        ),
        "late_fee": o.calculate_late_fee(),
        "status": "คืนแล้ว" if o.return_date else "ค้างส่ง",
    }
    for o in all_demo_orders
]

demo_summary_df = pd.DataFrame(summary_rows)
demo_summary_df

,loan_id,member_id,member_name,book_title,borrow_date,due_date,return_date,late_fee,status
0,L001,PS0001,รติ,พิธีกรรมและประเพณีทางศาสนา,2025-01-20,2025-02-03,2025-02-01,0,คืนแล้ว
1,L001,PS0034,โอโซน,การเขียนบทความสร้างสรรค์,2025-01-26,2025-02-09,ยังไม่คืน,0,ค้างส่ง
2,L002,PT0011,ภา,อาณาจักรโบราณในดินแดนไทย,2025-01-04,2025-02-03,ยังไม่คืน,0,ค้างส่ง
3,L003,PS0185,ชะเอม,ประวัติศาสตร์ไทยเบื้องต้น,2025-01-01,2025-01-15,ยังไม่คืน,0,ค้างส่ง
4,L004,PS0213,มัชฌิ์,เคมีอินทรีย์เบื้องต้น,2025-01-11,2025-01-25,2025-01-21,0,คืนแล้ว
5,L005,PS0201,นุ่น,การพัฒนาเว็บไซต์สมัยใหม่,2025-01-21,2025-02-04,2025-02-06,10,คืนแล้ว


In [508]:
import random
from datetime import datetime, timedelta
import pandas as pd


# 1. ฟังก์ชันจำลองการยืมแบบไม่พิมพ์ข้อความ (สำหรับ Batch Large-scale)

def simulate_borrow_batch(
    num_records=300, start_date=datetime(2026, 1, 1), days_range=180
):
    """จำลองการยืมหนังสือจำนวนหลายรายการแบบรวดเร็ว (ไม่มีการ print) -> คืนค่าเป็น DataFrame"""
    members = load_members("members.csv")
    books = load_books("books.csv")

    records = []

    for i in range(1, num_records + 1):
        loan_id = f"L{i:04d}"

        # สุ่มสมาชิก หนังสือ และวันที่ยืม
        member = random.choice(members)
        book = random.choice(books)
        borrow_date = start_date + timedelta(
            days=random.randint(0, days_range)
        )

        # สร้าง Object และคำนวณการคืน
        loan = BookLoan(loan_id, book, member, borrow_date)
        outcome = decide_return_outcome()
        return_date = calculate_return_date(loan.due_date, outcome)

        if return_date is not None:
            loan.mark_returned(return_date)

        # เก็บข้อมูลลงใน dict เพื่อแปลงเป็น DataFrame
        records.append({
            "loan_id": loan.loan_id,
            "member_id": loan.member.member_id,
            "member_name": loan.member.name,
            "member_type": loan.member.member_type,
            "book_id": loan.book.book_id,
            "book_title": loan.book.title,
            "borrow_date": loan.borrow_date.strftime("%Y-%m-%d"),
            "due_date": loan.due_date.strftime("%Y-%m-%d"),
            "return_date": (
                loan.return_date.strftime("%Y-%m-%d")
                if loan.return_date
                else None
            ),
            "late_fee": loan.calculate_late_fee(),
            "status": "คืนแล้ว" if loan.return_date else "ค้างส่ง",
        })

    return pd.DataFrame(records)



# 2. เรียกใช้งานจำลอง 300 รายการ และแสดงผลสรุป

df_loans_300 = simulate_borrow_batch(num_records=300)

# แสดงตัวอย่างข้อมูล 5 รายการแรก
print(f"✅ จำลองข้อมูลสำเร็จทั้งหมด {len(df_loans_300)} รายการ\n")
df_loans_300.head()

✅ จำลองข้อมูลสำเร็จทั้งหมด 300 รายการ



,loan_id,member_id,member_name,member_type,book_id,book_title,borrow_date,due_date,return_date,late_fee,status
0,L0001,PS0233,เฟิร์ส,นักศึกษา,D900-009,ประวัติศาสตร์อารยธรรมจีน,2026-04-16,2026-04-30,None,0,ค้างส่ง
1,L0002,PS0064,ไบร์ท,นักศึกษา,D500-002,เคมีทั่วไป,2026-06-07,2026-06-21,2026-06-16,0,คืนแล้ว
2,L0003,PS0038,แป้ง,นักศึกษา,D100-014,จิตวิเคราะห์เบื้องต้น,2026-03-03,2026-03-17,2026-03-23,30,คืนแล้ว
3,L0004,PT0040,ป๋อง,อาจารย์,D400-015,พจนานุกรมศัพท์ทางวิชาการ,2026-02-08,2026-03-10,2026-03-12,10,คืนแล้ว
4,L0005,PS0067,น้ำหวาน,นักศึกษา,D800-006,วรรณคดีโลกเบื้องต้น,2026-03-26,2026-04-09,2026-04-07,0,คืนแล้ว


In [491]:
# 5.แปลงข้อมูลเป็น DataFrame และบันทึก CSV
records = [
    {
        "loan_id": loan.loan_id,
        "book_id": loan.book.book_id,
        "member_id": loan.member.member_id,
        "borrow_date": loan.borrow_date.strftime("%Y-%m-%d"),
        "due_date": loan.due_date.strftime("%Y-%m-%d"),
        "return_date": (
            loan.return_date.strftime("%Y-%m-%d")
            if loan.return_date
            else None
        ),
        "late_fee": loan.calculate_late_fee()
    }

    for loan in loans
]


df_loans = pd.DataFrame(records)

df_loans.to_csv(
    "loans.csv",
    index=False
)

print("บันทึกไฟล์ CSV เสร็จ ขนาด:", df_loans.shape)

df_loans.head()

บันทึกไฟล์ CSV เสร็จ ขนาด: (350, 7)


,loan_id,book_id,member_id,borrow_date,due_date,return_date,late_fee
0,L0001,D200-005,PS0292,2025-02-02,2025-02-16,2025-02-24,40
1,L0002,D700-011,PS0242,2025-07-14,2025-07-28,2025-07-28,0
2,L0003,D800-005,PS0015,2025-07-19,2025-08-02,2025-08-03,5
3,L0004,D700-010,PS0137,2025-04-28,2025-05-12,2025-05-12,0
4,L0005,D500-007,PS0016,2025-01-12,2025-01-26,None,0


# ขั้นที่ 4 — จำลองข้อมูลทีละรายการด้วย Loop

In [473]:
# จำลองการใช้บริการยืม-คืนหนังสือของนักศึกษา จำนวน 3000 คน

def borrow(loan_id, member_id, book_id, borrow_date):
    if member_id not in members.index:
        print(f"ไม่พบ member_id '{member_id}'")
        return
    if book_id not in books.index:
        print(f"ไม่พบ book_id '{book_id}'")
        return
    name = members.loc[member_id, "name"]
    title = books.loc[book_id, "title"]
    loan_period = int(members.loc[member_id, "loan_period_days"])
    due_date = borrow_date + timedelta(days=loan_period)
    print(f"{loan_id}: นักศึกษา: {name} ({member_id}) ยืมหนังสือ: {title} ({book_id}) | วันที่ยืม: {borrow_date.date()} | กำหนดคืน: {due_date.date()}")


In [474]:
# รายการที่ 0001
borrow("1", "PS0001", "D000-001", datetime(2025, 1, 1))

TypeError: argument of type 'builtin_function_or_method' is not iterable

In [ ]:
borrow("2", "PS0002", "D000-002", datetime(2025, 1, 2))

In [ ]:
borrow("3", "PS0003", "D000-003", datetime(2025, 1, 3))

In [ ]:
borrow("4", "PS0004", "D000-004", datetime(2025, 1, 4))

In [ ]:
borrow("5", "PS0005", "D000-005", datetime(2025, 1, 5))

In [ ]:
borrow("6", "PS0006", "D000-006", datetime(2025, 1, 6))

In [ ]:
borrow("7", "PS0007", "D000-007", datetime(2025, 1, 7))

In [ ]:
borrow("8", "PS0008", "D000-008", datetime(2025, 1, 8))

In [ ]:
borrow("9", "PS0009", "D000-009", datetime(2025, 1, 9))

In [ ]:
borrow("10", "PS0010", "D000-010", datetime(2025, 1, 10))

In [ ]:
borrow("11", "PS0011", "D000-011", datetime(2025, 1, 11))

In [ ]:
borrow("12", "PS0012", "D000-012", datetime(2025, 1, 12))

In [ ]:
borrow("13", "PS0013", "D000-013", datetime(2025, 1, 13))

In [ ]:
borrow("14", "PS0014", "D000-014", datetime(2025, 1, 14))

In [ ]:
borrow("15", "PS0015", "D000-015", datetime(2025, 1, 15))

In [ ]:
borrow("16", "PS0016", "D100-001", datetime(2025, 1, 16))

In [ ]:
borrow("17", "PS0017", "D100-002", datetime(2025, 1, 17))

In [ ]:
borrow("18", "PS0018", "D100-003", datetime(2025, 1, 18))

In [ ]:
borrow("19", "PS0019", "D100-004", datetime(2025, 1, 19))

In [ ]:
borrow("20", "PS0020", "D100-005", datetime(2025, 1, 20))

In [ ]:
borrow("21", "PS0021", "D100-006", datetime(2025, 1, 21))

In [ ]:
borrow("22", "PS0022", "D100-007", datetime(2025, 1, 22))

In [ ]:
borrow("23", "PS0023", "D100-008", datetime(2025, 1, 23))

In [ ]:
borrow("24", "PS0024", "D100-009", datetime(2025, 1, 24))

In [ ]:
borrow("25", "PS0025", "D100-010", datetime(2025, 1, 25))

In [ ]:
borrow("26", "PS0026", "D100-011", datetime(2025, 1, 26))

In [ ]:
borrow("27", "PS0027", "D100-012", datetime(2025, 1, 27))

In [ ]:
borrow("28", "PS0028", "D100-013", datetime(2025, 1, 28))

In [ ]:
borrow("29", "PS0029", "D100-014", datetime(2025, 1, 29))

In [ ]:
borrow("30", "PS0030", "D100-015", datetime(2025, 1, 30))

In [ ]:
borrow("31", "PS0031", "D200-001", datetime(2025, 1, 31))

In [ ]:
borrow("32", "PS0032", "D200-002", datetime(2025, 2, 1))

In [ ]:
borrow("33", "PS0033", "D200-003", datetime(2025, 2, 2))

In [ ]:
borrow("34", "PS0034", "D200-004", datetime(2025, 2, 3))

In [ ]:
borrow("35", "PS0035", "D200-005", datetime(2025, 2, 4))

In [ ]:
borrow("36", "PS0036", "D200-006", datetime(2025, 2, 5))

In [ ]:
borrow("37", "PS0037", "D200-007", datetime(2025, 2, 6))

In [ ]:
borrow("38", "PS0038", "D200-008", datetime(2025, 2, 7))

In [ ]:
borrow("39", "PS0039", "D200-009", datetime(2025, 2, 8))

In [ ]:
borrow("40", "PS0040", "D200-010", datetime(2025, 2, 9))

In [ ]:
borrow("41", "PS0041", "D200-011", datetime(2025, 2, 10))

In [ ]:
borrow("42", "PS0042", "D200-012", datetime(2025, 2, 11))

In [ ]:
borrow("43", "PS0043", "D200-013", datetime(2025, 2, 12))

In [ ]:
borrow("44", "PS0044", "D200-014", datetime(2025, 2, 13))

In [ ]:
borrow("45", "PS0045", "D200-015", datetime(2025, 2, 14))

In [ ]:
borrow("46", "PS0046", "D300-001", datetime(2025, 2, 15))

In [ ]:
borrow("47", "PS0047", "D300-002", datetime(2025, 2, 16))

In [ ]:
borrow("48", "PS0048", "D300-003", datetime(2025, 2, 17))

In [ ]:
borrow("49", "PS0049", "D300-004", datetime(2025, 2, 18))

In [ ]:
borrow("50", "PS0050", "D300-005", datetime(2025, 2, 19))

In [ ]:
borrow("51", "PS0051", "D300-006", datetime(2025, 2, 20))

In [ ]:
borrow("52", "PS0052", "D300-007", datetime(2025, 2, 21))

In [ ]:
borrow("53", "PS0053", "D300-008", datetime(2025, 2, 22))

In [ ]:
borrow("54", "PS0054", "D300-009", datetime(2025, 2, 23))

In [ ]:
borrow("55", "PS0055", "D300-010", datetime(2025, 2, 24))

In [ ]:
borrow("56", "PS0056", "D300-011", datetime(2025, 2, 25))

In [ ]:
borrow("57", "PS0057", "D300-012", datetime(2025, 2, 26))

In [ ]:
borrow("58", "PS0058", "D300-013", datetime(2025, 2, 27))

In [ ]:
borrow("59", "PS0059", "D300-014", datetime(2025, 2, 28))

In [ ]:
borrow("60", "PS0060", "D300-015", datetime(2025, 3, 1))

In [ ]:
borrow("61", "PS0061", "D400-001", datetime(2025, 3, 2))

In [ ]:
borrow("62", "PS0062", "D400-002", datetime(2025, 3, 3))

In [ ]:
borrow("63", "PS0063", "D400-003", datetime(2025, 3, 4))

In [ ]:
borrow("64", "PS0064", "D400-004", datetime(2025, 3, 5))

In [ ]:
borrow("65", "PS0065", "D400-005", datetime(2025, 3, 6))

In [ ]:
borrow("66", "PS0066", "D400-006", datetime(2025, 3, 7))

In [ ]:
borrow("67", "PS0067", "D400-007", datetime(2025, 3, 8))

In [ ]:
borrow("68", "PS0068", "D400-008", datetime(2025, 3, 9))

In [ ]:
borrow("69", "PS0069", "D400-009", datetime(2025, 3, 10))

In [ ]:
borrow("70", "PS0070", "D400-010", datetime(2025, 3, 11))

In [ ]:
borrow("71", "PS0071", "D400-011", datetime(2025, 3, 12))

In [ ]:
borrow("72", "PS0072", "D400-012", datetime(2025, 3, 13))

In [ ]:
borrow("73", "PS0073", "D400-013", datetime(2025, 3, 14))

In [ ]:
borrow("74", "PS0074", "D400-014", datetime(2025, 3, 15))

In [ ]:
borrow("75", "PS0075", "D400-015", datetime(2025, 3, 16))

In [ ]:
borrow("76", "PS0076", "D500-001", datetime(2025, 3, 17))

In [ ]:
borrow("77", "PS0077", "D500-002", datetime(2025, 3, 18))

In [ ]:
borrow("78", "PS0078", "D500-003", datetime(2025, 3, 19))

In [ ]:
borrow("79", "PS0079", "D500-004", datetime(2025, 3, 20))

In [ ]:
borrow("80", "PS0080", "D500-005", datetime(2025, 3, 21))

In [ ]:
borrow("81", "PS0081", "D500-006", datetime(2025, 3, 22))

In [ ]:
borrow("82", "PS0082", "D500-007", datetime(2025, 3, 23))

In [ ]:
borrow("83", "PS0083", "D500-008", datetime(2025, 3, 24))

In [ ]:
borrow("84", "PS0084", "D500-009", datetime(2025, 3, 25))

In [ ]:
borrow("85", "PS0085", "D500-010", datetime(2025, 3, 26))

In [ ]:
borrow("86", "PS0086", "D500-011", datetime(2025, 3, 27))

In [ ]:
borrow("87", "PS0087", "D500-012", datetime(2025, 3, 28))

In [ ]:
borrow("88", "PS0088", "D500-013", datetime(2025, 3, 29))

In [ ]:
borrow("89", "PS0089", "D500-014", datetime(2025, 3, 30))

In [ ]:
borrow("90", "PS0090", "D500-015", datetime(2025, 3, 31))

In [ ]:
borrow("91", "PS0091", "D600-001", datetime(2025, 4, 1))

In [ ]:
borrow("92", "PS0092", "D600-002", datetime(2025, 4, 2))

In [ ]:
borrow("93", "PS0093", "D600-003", datetime(2025, 4, 3))

In [ ]:
borrow("94", "PS0094", "D600-004", datetime(2025, 4, 4))

In [ ]:
borrow("95", "PS0095", "D600-005", datetime(2025, 4, 5))

In [ ]:
borrow("96", "PS0096", "D600-006", datetime(2025, 4, 6))

In [ ]:
borrow("97", "PS0097", "D600-007", datetime(2025, 4, 7))

In [ ]:
borrow("98", "PS0098", "D600-008", datetime(2025, 4, 8))

In [ ]:
borrow("99", "PS0099", "D600-009", datetime(2025, 4, 9))

In [ ]:
borrow("100", "PS0100", "D600-010", datetime(2025, 4, 10))

In [ ]:
borrow("101", "PS0101", "D600-011", datetime(2025, 4, 11))

In [ ]:
borrow("102", "PS0102", "D600-012", datetime(2025, 4, 12))

In [ ]:
borrow("103", "PS0103", "D600-013", datetime(2025, 4, 13))

In [ ]:
borrow("104", "PS0104", "D600-014", datetime(2025, 4, 14))

In [ ]:
borrow("105", "PS0105", "D600-015", datetime(2025, 4, 15))

In [ ]:
borrow("106", "PS0106", "D700-001", datetime(2025, 4, 16))

In [ ]:
borrow("107", "PS0107", "D700-002", datetime(2025, 4, 17))

In [ ]:
borrow("108", "PS0108", "D700-003", datetime(2025, 4, 18))

In [ ]:
borrow("109", "PS0109", "D700-004", datetime(2025, 4, 19))

In [ ]:
borrow("110", "PS0110", "D700-005", datetime(2025, 4, 20))

In [ ]:
borrow("111", "PS0111", "D700-006", datetime(2025, 4, 21))

In [ ]:
borrow("112", "PS0112", "D700-007", datetime(2025, 4, 22))

In [ ]:
borrow("113", "PS0113", "D700-008", datetime(2025, 4, 23))

In [ ]:
borrow("114", "PS0114", "D700-009", datetime(2025, 4, 24))

In [ ]:
borrow("115", "PS0115", "D700-010", datetime(2025, 4, 25))

In [ ]:
borrow("116", "PS0116", "D700-011", datetime(2025, 4, 26))

In [ ]:
borrow("117", "PS0117", "D700-012", datetime(2025, 4, 27))

In [ ]:
borrow("118", "PS0118", "D700-013", datetime(2025, 4, 28))

In [ ]:
borrow("119", "PS0119", "D700-014", datetime(2025, 4, 29))

In [ ]:
borrow("120", "PS0120", "D700-015", datetime(2025, 4, 30))

In [ ]:
borrow("121", "PS0121", "D800-001", datetime(2025, 5, 1))

In [ ]:
borrow("122", "PS0122", "D800-002", datetime(2025, 5, 2))

In [ ]:
borrow("123", "PS0123", "D800-003", datetime(2025, 5, 3))

In [ ]:
borrow("124", "PS0124", "D800-004", datetime(2025, 5, 4))

In [ ]:
borrow("125", "PS0125", "D800-005", datetime(2025, 5, 5))

In [ ]:
borrow("126", "PS0126", "D800-006", datetime(2025, 5, 6))

In [ ]:
borrow("127", "PS0127", "D800-007", datetime(2025, 5, 7))

In [ ]:
borrow("128", "PS0128", "D800-008", datetime(2025, 5, 8))

In [ ]:
borrow("129", "PS0129", "D800-009", datetime(2025, 5, 9))

In [ ]:
borrow("130", "PS0130", "D800-010", datetime(2025, 5, 10))

In [ ]:
borrow("131", "PS0131", "D800-011", datetime(2025, 5, 11))

In [ ]:
borrow("132", "PS0132", "D800-012", datetime(2025, 5, 12))

In [ ]:
borrow("133", "PS0133", "D800-013", datetime(2025, 5, 13))

In [ ]:
borrow("134", "PS0134", "D800-014", datetime(2025, 5, 14))

In [ ]:
borrow("135", "PS0135", "D800-015", datetime(2025, 5, 15))

In [ ]:
borrow("136", "PS0136", "D900-001", datetime(2025, 5, 16))

In [ ]:
borrow("137", "PS0137", "D900-002", datetime(2025, 5, 17))

In [ ]:
borrow("138", "PS0138", "D900-003", datetime(2025, 5, 18))

In [ ]:
borrow("139", "PS0139", "D900-004", datetime(2025, 5, 19))

In [ ]:
borrow("140", "PS0140", "D900-005", datetime(2025, 5, 20))

In [ ]:
borrow("141", "PS0141", "D900-006", datetime(2025, 5, 21))

In [ ]:
borrow("142", "PS0142", "D900-007", datetime(2025, 5, 22))

In [ ]:
borrow("143", "PS0143", "D900-008", datetime(2025, 5, 23))

In [ ]:
borrow("144", "PS0144", "D900-009", datetime(2025, 5, 24))

In [ ]:
borrow("145", "PS0145", "D900-010", datetime(2025, 5, 25))

In [ ]:
borrow("146", "PS0146", "D900-011", datetime(2025, 5, 26))

In [ ]:
borrow("147", "PS0147", "D900-012", datetime(2025, 5, 27))

In [ ]:
borrow("148", "PS0148", "D900-013", datetime(2025, 5, 28))

In [ ]:
borrow("149", "PS0149", "D900-014", datetime(2025, 5, 29))

In [ ]:
borrow("150", "PS0150", "D900-015", datetime(2025, 5, 30))

In [ ]:
borrow("151", "PS0151", "D000-001", datetime(2025, 5, 31))

In [ ]:
borrow("152", "PS0152", "D000-002", datetime(2025, 6, 1))

In [ ]:
borrow("153", "PS0153", "D000-003", datetime(2025, 6, 2))

In [ ]:
borrow("154", "PS0154", "D000-004", datetime(2025, 6, 3))

In [ ]:
borrow("155", "PS0155", "D000-005", datetime(2025, 6, 4))

In [ ]:
borrow("156", "PS0156", "D000-006", datetime(2025, 6, 5))

In [ ]:
borrow("157", "PS0157", "D000-007", datetime(2025, 6, 6))

In [ ]:
borrow("158", "PS0158", "D000-008", datetime(2025, 6, 7))

In [ ]:
borrow("159", "PS0159", "D000-009", datetime(2025, 6, 8))

In [ ]:
borrow("160", "PS0160", "D000-010", datetime(2025, 6, 9))

In [ ]:
borrow("161", "PS0161", "D000-011", datetime(2025, 6, 10))

In [ ]:
borrow("162", "PS0162", "D000-012", datetime(2025, 6, 11))

In [ ]:
borrow("163", "PS0163", "D000-013", datetime(2025, 6, 12))

In [ ]:
borrow("164", "PS0164", "D000-014", datetime(2025, 6, 13))

In [ ]:
borrow("165", "PS0165", "D000-015", datetime(2025, 6, 14))

In [ ]:
borrow("166", "PS0166", "D100-001", datetime(2025, 6, 15))

In [ ]:
borrow("167", "PS0167", "D100-002", datetime(2025, 6, 16))

In [ ]:
borrow("168", "PS0168", "D100-003", datetime(2025, 6, 17))

In [ ]:
borrow("169", "PS0169", "D100-004", datetime(2025, 6, 18))

In [ ]:
borrow("170", "PS0170", "D100-005", datetime(2025, 6, 19))

In [ ]:
borrow("171", "PS0171", "D100-006", datetime(2025, 6, 20))

In [ ]:
borrow("172", "PS0172", "D100-007", datetime(2025, 6, 21))

In [ ]:
borrow("173", "PS0173", "D100-008", datetime(2025, 6, 22))

In [ ]:
borrow("174", "PS0174", "D100-009", datetime(2025, 6, 23))

In [ ]:
borrow("175", "PS0175", "D100-010", datetime(2025, 6, 24))

In [ ]:
borrow("176", "PS0176", "D100-011", datetime(2025, 6, 25))

In [ ]:
borrow("177", "PS0177", "D100-012", datetime(2025, 6, 26))

In [ ]:
borrow("178", "PS0178", "D100-013", datetime(2025, 6, 27))

In [ ]:
borrow("179", "PS0179", "D100-014", datetime(2025, 6, 28))

In [ ]:
borrow("180", "PS0180", "D100-015", datetime(2025, 6, 29))

In [ ]:
borrow("181", "PS0181", "D200-001", datetime(2025, 6, 30))

In [ ]:
borrow("182", "PS0182", "D200-002", datetime(2025, 7, 1))

In [ ]:
borrow("183", "PS0183", "D200-003", datetime(2025, 7, 2))

In [ ]:
borrow("184", "PS0184", "D200-004", datetime(2025, 7, 3))

In [ ]:
borrow("185", "PS0185", "D200-005", datetime(2025, 7, 4))

In [ ]:
borrow("186", "PS0186", "D200-006", datetime(2025, 7, 5))

In [ ]:
borrow("187", "PS0187", "D200-007", datetime(2025, 7, 6))

In [ ]:
borrow("188", "PS0188", "D200-008", datetime(2025, 7, 7))

In [ ]:
borrow("189", "PS0189", "D200-009", datetime(2025, 7, 8))

In [ ]:
borrow("190", "PS0190", "D200-010", datetime(2025, 7, 9))

In [ ]:
borrow("191", "PS0191", "D200-011", datetime(2025, 7, 10))

In [ ]:
borrow("192", "PS0192", "D200-012", datetime(2025, 7, 11))

In [ ]:
borrow("193", "PS0193", "D200-013", datetime(2025, 7, 12))

In [ ]:
borrow("194", "PS0194", "D200-014", datetime(2025, 7, 13))

In [ ]:
borrow("195", "PS0195", "D200-015", datetime(2025, 7, 14))

In [ ]:
borrow("196", "PS0196", "D300-001", datetime(2025, 7, 15))

In [ ]:
borrow("197", "PS0197", "D300-002", datetime(2025, 7, 16))

In [ ]:
borrow("198", "PS0198", "D300-003", datetime(2025, 7, 17))

In [ ]:
borrow("199", "PS0199", "D300-004", datetime(2025, 7, 18))

In [ ]:
borrow("200", "PS0200", "D300-005", datetime(2025, 7, 19))

In [ ]:
borrow("201", "PS0201", "D300-006", datetime(2025, 7, 20))

In [ ]:
borrow("202", "PS0202", "D300-007", datetime(2025, 7, 21))

In [ ]:
borrow("203", "PS0203", "D300-008", datetime(2025, 7, 22))

In [ ]:
borrow("204", "PS0204", "D300-009", datetime(2025, 7, 23))

In [ ]:
borrow("205", "PS0205", "D300-010", datetime(2025, 7, 24))

In [ ]:
borrow("206", "PS0206", "D300-011", datetime(2025, 7, 25))

In [ ]:
borrow("207", "PS0207", "D300-012", datetime(2025, 7, 26))

In [ ]:
borrow("208", "PS0208", "D300-013", datetime(2025, 7, 27))

In [ ]:
borrow("209", "PS0209", "D300-014", datetime(2025, 7, 28))

In [ ]:
borrow("210", "PS0210", "D300-015", datetime(2025, 7, 29))

In [ ]:
borrow("211", "PS0211", "D400-001", datetime(2025, 7, 30))

In [ ]:
borrow("212", "PS0212", "D400-002", datetime(2025, 7, 31))

In [ ]:
borrow("213", "PS0213", "D400-003", datetime(2025, 8, 1))

In [ ]:
borrow("214", "PS0214", "D400-004", datetime(2025, 8, 2))

In [ ]:
borrow("215", "PS0215", "D400-005", datetime(2025, 8, 3))

In [ ]:
borrow("216", "PS0216", "D400-006", datetime(2025, 8, 4))

In [ ]:
borrow("217", "PS0217", "D400-007", datetime(2025, 8, 5))

In [ ]:
borrow("218", "PS0218", "D400-008", datetime(2025, 8, 6))

In [ ]:
borrow("219", "PS0219", "D400-009", datetime(2025, 8, 7))

In [ ]:
borrow("220", "PS0220", "D400-010", datetime(2025, 8, 8))

In [ ]:
borrow("221", "PS0221", "D400-011", datetime(2025, 8, 9))

In [ ]:
borrow("222", "PS0222", "D400-012", datetime(2025, 8, 10))

In [ ]:
borrow("223", "PS0223", "D400-013", datetime(2025, 8, 11))

In [ ]:
borrow("224", "PS0224", "D400-014", datetime(2025, 8, 12))

In [ ]:
borrow("225", "PS0225", "D400-015", datetime(2025, 8, 13))

In [ ]:
borrow("226", "PS0226", "D500-001", datetime(2025, 8, 14))

In [ ]:
borrow("227", "PS0227", "D500-002", datetime(2025, 8, 15))

In [ ]:
borrow("228", "PS0228", "D500-003", datetime(2025, 8, 16))

In [ ]:
borrow("229", "PS0229", "D500-004", datetime(2025, 8, 17))

In [ ]:
borrow("230", "PS0230", "D500-005", datetime(2025, 8, 18))

In [ ]:
borrow("231", "PS0231", "D500-006", datetime(2025, 8, 19))

In [ ]:
borrow("232", "PS0232", "D500-007", datetime(2025, 8, 20))

In [ ]:
borrow("233", "PS0233", "D500-008", datetime(2025, 8, 21))

In [ ]:
borrow("234", "PS0234", "D500-009", datetime(2025, 8, 22))

In [ ]:
borrow("235", "PS0235", "D500-010", datetime(2025, 8, 23))

In [ ]:
borrow("236", "PS0236", "D500-011", datetime(2025, 8, 24))

In [ ]:
borrow("237", "PS0237", "D500-012", datetime(2025, 8, 25))

In [ ]:
borrow("238", "PS0238", "D500-013", datetime(2025, 8, 26))

In [ ]:
borrow("239", "PS0239", "D500-014", datetime(2025, 8, 27))

In [ ]:
borrow("240", "PS0240", "D500-015", datetime(2025, 8, 28))

In [ ]:
borrow("241", "PS0241", "D600-001", datetime(2025, 8, 29))

In [ ]:
borrow("242", "PS0242", "D600-002", datetime(2025, 8, 30))

In [ ]:
borrow("243", "PS0243", "D600-003", datetime(2025, 8, 31))

In [ ]:
borrow("244", "PS0244", "D600-004", datetime(2025, 9, 1))

In [ ]:
borrow("245", "PS0245", "D600-005", datetime(2025, 9, 2))

In [ ]:
borrow("246", "PS0246", "D600-006", datetime(2025, 9, 3))

In [ ]:
borrow("247", "PS0247", "D600-007", datetime(2025, 9, 4))

In [ ]:
borrow("248", "PS0248", "D600-008", datetime(2025, 9, 5))

In [ ]:
borrow("249", "PS0249", "D600-009", datetime(2025, 9, 6))

In [ ]:
borrow("250", "PS0250", "D600-010", datetime(2025, 9, 7))

In [ ]:
borrow("251", "PS0251", "D600-011", datetime(2025, 9, 8))

In [ ]:
borrow("252", "PS0252", "D600-012", datetime(2025, 9, 9))

In [ ]:
borrow("253", "PS0253", "D600-013", datetime(2025, 9, 10))

In [ ]:
borrow("254", "PS0254", "D600-014", datetime(2025, 9, 11))

In [ ]:
borrow("255", "PS0255", "D600-015", datetime(2025, 9, 12))

In [ ]:
borrow("256", "PS0256", "D700-001", datetime(2025, 9, 13))

In [ ]:
borrow("257", "PS0257", "D700-002", datetime(2025, 9, 14))

In [ ]:
borrow("258", "PS0258", "D700-003", datetime(2025, 9, 15))

In [ ]:
borrow("259", "PS0259", "D700-004", datetime(2025, 9, 16))

In [ ]:
borrow("260", "PS0260", "D700-005", datetime(2025, 9, 17))

In [ ]:
borrow("261", "PS0261", "D700-006", datetime(2025, 9, 18))

In [ ]:
borrow("262", "PS0262", "D700-007", datetime(2025, 9, 19))

In [ ]:
borrow("263", "PS0263", "D700-008", datetime(2025, 9, 20))

In [ ]:
borrow("264", "PS0264", "D700-009", datetime(2025, 9, 21))

In [ ]:
borrow("265", "PS0265", "D700-010", datetime(2025, 9, 22))

In [ ]:
borrow("266", "PS0266", "D700-011", datetime(2025, 9, 23))

In [ ]:
borrow("267", "PS0267", "D700-012", datetime(2025, 9, 24))

In [ ]:
borrow("268", "PS0268", "D700-013", datetime(2025, 9, 25))

In [ ]:
borrow("269", "PS0269", "D700-014", datetime(2025, 9, 26))

In [ ]:
borrow("270", "PS0270", "D700-015", datetime(2025, 9, 27))

In [ ]:
borrow("271", "PS0271", "D800-001", datetime(2025, 9, 28))

In [ ]:
borrow("272", "PS0272", "D800-002", datetime(2025, 9, 29))

In [ ]:
borrow("273", "PS0273", "D800-003", datetime(2025, 9, 30))

In [ ]:
borrow("274", "PS0274", "D800-004", datetime(2025, 10, 1))

In [ ]:
borrow("275", "PS0275", "D800-005", datetime(2025, 10, 2))

In [ ]:
borrow("276", "PS0276", "D800-006", datetime(2025, 10, 3))

In [ ]:
borrow("277", "PS0277", "D800-007", datetime(2025, 10, 4))

In [ ]:
borrow("278", "PS0278", "D800-008", datetime(2025, 10, 5))

In [ ]:
borrow("279", "PS0279", "D800-009", datetime(2025, 10, 6))

In [ ]:
borrow("280", "PS0280", "D800-010", datetime(2025, 10, 7))

In [ ]:
borrow("281", "PS0281", "D800-011", datetime(2025, 10, 8))

In [ ]:
borrow("282", "PS0282", "D800-012", datetime(2025, 10, 9))

In [ ]:
borrow("283", "PS0283", "D800-013", datetime(2025, 10, 10))

In [ ]:
borrow("284", "PS0284", "D800-014", datetime(2025, 10, 11))

In [ ]:
borrow("285", "PS0285", "D800-015", datetime(2025, 10, 12))

In [ ]:
borrow("286", "PS0286", "D900-001", datetime(2025, 10, 13))

In [ ]:
borrow("287", "PS0287", "D900-002", datetime(2025, 10, 14))

In [ ]:
borrow("288", "PS0288", "D900-003", datetime(2025, 10, 15))

In [ ]:
borrow("289", "PS0289", "D900-004", datetime(2025, 10, 16))

In [ ]:
borrow("290", "PS0290", "D900-005", datetime(2025, 10, 17))

In [ ]:
borrow("291", "PS0291", "D900-006", datetime(2025, 10, 18))

In [ ]:
borrow("292", "PS0292", "D900-007", datetime(2025, 10, 19))

In [ ]:
borrow("293", "PS0293", "D900-008", datetime(2025, 10, 20))

In [ ]:
borrow("294", "PS0294", "D900-009", datetime(2025, 10, 21))

In [ ]:
borrow("295", "PS0295", "D900-010", datetime(2025, 10, 22))

In [ ]:
borrow("296", "PS0296", "D900-011", datetime(2025, 10, 23))

In [ ]:
borrow("297", "PS0297", "D900-012", datetime(2025, 10, 24))

In [ ]:
borrow("298", "PS0298", "D900-013", datetime(2025, 10, 25))

In [ ]:
borrow("299", "PS0299", "D900-014", datetime(2025, 10, 26))

In [ ]:
borrow("300", "PS0300", "D900-015", datetime(2025, 10, 27))

In [ ]:
# จำลองการใช้บริการยืม-คืนหนังสือของอาจารย์ จำนวน 50 คน
def borrow(loan_id, member_id, book_id, borrow_date):
    if member_id not in members.index:
        print(f"ไม่พบ member_id '{member_id}'")
        return
    if book_id not in books.index:
        print(f"ไม่พบ book_id '{book_id}'")
        return
    name = members.loc[member_id, "name"]
    title = books.loc[book_id, "title"]
    loan_period = int(members.loc[member_id, "loan_period_days"])
    due_date = borrow_date + timedelta(days=loan_period)
    print(f"{loan_id}: อาจารย์: {name} ({member_id}) ยืมหนังสือ: {title} ({book_id}) | วันที่ยืม: {borrow_date.date()} | กำหนดคืน: {due_date.date()}")

In [ ]:
# รายการที่ 0001
borrow("301", "PT0001", "D000-001", datetime(2025, 10, 28))

In [ ]:
borrow("302", "PT0002", "D000-002", datetime(2025, 10, 29))

In [ ]:
borrow("303", "PT0003", "D000-003", datetime(2025, 10, 30))

In [ ]:
borrow("304", "PT0004", "D000-004", datetime(2025, 10, 31))

In [ ]:
borrow("305", "PT0005", "D000-005", datetime(2025, 11, 1))

In [ ]:
borrow("306", "PT0006", "D000-006", datetime(2025, 11, 2))

In [ ]:
borrow("307", "PT0007", "D000-007", datetime(2025, 11, 3))

In [ ]:
borrow("308", "PT0008", "D000-008", datetime(2025, 11, 4))

In [ ]:
borrow("309", "PT0009", "D000-009", datetime(2025, 11, 5))

In [ ]:
borrow("310", "PT0010", "D000-010", datetime(2025, 11, 6))

In [ ]:
borrow("311", "PT0011", "D000-011", datetime(2025, 11, 7))

In [ ]:
borrow("312", "PT0012", "D000-012", datetime(2025, 11, 8))

In [ ]:
borrow("313", "PT0013", "D000-013", datetime(2025, 11, 9))

In [ ]:
borrow("314", "PT0014", "D000-014", datetime(2025, 11, 10))

In [ ]:
borrow("315", "PT0015", "D000-015", datetime(2025, 11, 11))

In [ ]:
borrow("316", "PT0016", "D100-001", datetime(2025, 11, 12))

In [ ]:
borrow("317", "PT0017", "D100-002", datetime(2025, 11, 13))

In [ ]:
borrow("318", "PT0018", "D100-003", datetime(2025, 11, 14))

In [ ]:
borrow("319", "PT0019", "D100-004", datetime(2025, 11, 15))

In [ ]:
borrow("320", "PT0020", "D100-005", datetime(2025, 11, 16))

In [ ]:
borrow("321", "PT0021", "D100-006", datetime(2025, 11, 17))

In [ ]:
borrow("322", "PT0022", "D100-007", datetime(2025, 11, 18))

In [ ]:
borrow("323", "PT0023", "D100-008", datetime(2025, 11, 19))

In [ ]:
borrow("324", "PT0024", "D100-009", datetime(2025, 11, 20))

In [ ]:
borrow("325", "PT0025", "D100-010", datetime(2025, 11, 21))

In [ ]:
borrow("326", "PT0026", "D100-011", datetime(2025, 11, 22))

In [ ]:
borrow("327", "PT0027", "D100-012", datetime(2025, 11, 23))

In [ ]:
borrow("328", "PT0028", "D100-013", datetime(2025, 11, 24))

In [ ]:
borrow("329", "PT0029", "D100-014", datetime(2025, 11, 25))

In [ ]:
borrow("330", "PT0030", "D100-015", datetime(2025, 11, 26))

In [ ]:
borrow("331", "PT0031", "D200-001", datetime(2025, 11, 27))

In [ ]:
borrow("332", "PT0032", "D200-002", datetime(2025, 11, 28))

In [ ]:
borrow("333", "PT0033", "D200-003", datetime(2025, 11, 29))

In [ ]:
borrow("334", "PT0034", "D200-004", datetime(2025, 11, 30))

In [ ]:
borrow("335", "PT0035", "D200-005", datetime(2025, 12, 1))

In [ ]:
borrow("336", "PT0036", "D200-006", datetime(2025, 12, 2))

In [ ]:
borrow("337", "PT0037", "D200-007", datetime(2025, 12, 3))

In [ ]:
borrow("338", "PT0038", "D200-008", datetime(2025, 12, 4))

In [ ]:
borrow("339", "PT0039", "D200-009", datetime(2025, 12, 5))

In [ ]:
borrow("340", "PT0040", "D200-010", datetime(2025, 12, 6))

In [ ]:
borrow("341", "PT0041", "D200-011", datetime(2025, 12, 7))

In [ ]:
borrow("342", "PT0042", "D200-012", datetime(2025, 12, 8))

In [ ]:
borrow("343", "PT0043", "D200-013", datetime(2025, 12, 9))

In [ ]:
borrow("344", "PT0044", "D200-014", datetime(2025, 12, 10))

In [ ]:
borrow("345", "PT0045", "D200-015", datetime(2025, 12, 11))

In [ ]:
borrow("346", "PT0046", "D300-001", datetime(2025, 12, 12))

In [ ]:
borrow("347", "PT0047", "D300-002", datetime(2025, 12, 13))

In [ ]:
borrow("348", "PT0048", "D300-003", datetime(2025, 12, 14))

In [ ]:
borrow("349", "PT0049", "D300-004", datetime(2025, 12, 15))

In [ ]:
# 5.แปลงข้อมูลเป็น DataFrame และบันทึก CSV
records = [
    {
        "loan_id": loan.loan_id,
        "book_id": loan.book.book_id,
        "member_id": loan.member.member_id,
        "borrow_date": loan.borrow_date.strftime("%Y-%m-%d"),
        "due_date": loan.due_date.strftime("%Y-%m-%d"),
        "return_date": (
            loan.return_date.strftime("%Y-%m-%d")
            if loan.return_date
            else None
        ),
        "late_fee": loan.calculate_late_fee()
    }

    for loan in loans
]


df_loans = pd.DataFrame(records)

df_loans.to_csv(
    "loans.csv",
    index=False
)

print("บันทึกไฟล์ CSV เสร็จ ขนาด:", df_loans.shape)

df_loans.head()

In [ ]:
borrow("350", "PT0050", "D300-005", datetime(2025, 12, 16))

In [ ]:
#3.จำลองการยืมหนังสือ 350 รายการ
random.seed(1)

books = load_books("books.csv")
members = load_members("members.csv")

loans = []

start_date = datetime(2025, 1, 1)


for i in range(1, 351):

    available_books = [
        b for b in books
        if b.is_available
    ]

    if not available_books:

        available_books = books

        for b in available_books:
            b.return_book()

    book = random.choice(available_books)

    member = random.choice(members)

    borrow_date = random_borrow_date(start_date)

    loan_id = f"L{i:04d}"

    loan = BookLoan(
        loan_id,
        book,
        member,
        borrow_date
    )

    book.borrow()

    outcome = decide_return_outcome()

    return_date = calculate_return_date(
        loan.due_date,
        outcome
    )

    if return_date is not None:
        loan.mark_returned(return_date)

    loans.append(loan)


print(f"จำลองการยืมเสร็จแล้วทั้งหมด {len(loans)} รายการ")


# ขั้นที่ 5 — แปลง Object เป็นตาราง แล้วบันทึกเป็น CSV

# ขั้นที่ 6 — สร้างฐานข้อมูล SQLite จากข้อมูลเดียวกัน

In [ ]:
# 6.สร้าง SQLite Database
import sqlite3

df_books = pd.read_csv("books.csv")
df_members = pd.read_csv("members.csv")

conn = sqlite3.connect("library.db")

df_books.to_sql(
    "books",
    conn,
    if_exists="replace",
    index=False
)

df_members.to_sql(
    "members",
    conn,
    if_exists="replace",
    index=False
)

df_loans.to_sql(
    "loans",
    conn,
    if_exists="replace",
    index=False
)


check = pd.read_sql_query(
    "SELECT * FROM loans LIMIT 5",
    conn
)

print(check)

# ขั้นที่ 7 — วิเคราะห์ข้อมูลด้วย pandas และ SQL

In [ ]:
# 7.วิเคราะห์ด้วย Pandas
df_merged = df_loans.merge(
    df_books[["book_id", "category_name"]],
    on="book_id",
    how="left"
)

df_merged = df_merged.merge(
    df_members[["member_id", "member_type", "name"]],
    on="member_id",
    how="left"
)

#คำถามที่ 1: หมวดไหนถูกยืมมากที่สุด?
category_summary = df_merged.groupby(
    "category_name"
).agg(
    total_loans=("loan_id", "count"),
    total_late_fee=("late_fee", "sum")
).reset_index().sort_values(
    "total_loans",
    ascending=False
)

print(category_summary)

#คำถามที่ 2: สมาชิกประเภทไหนคืนหนังสือช้ากว่า?
member_type_summary = df_merged.groupby(
    "member_type"
).agg(
    total_loans=("loan_id", "count"),
    avg_late_fee=("late_fee", "mean")
).reset_index().sort_values(
    "avg_late_fee",
    ascending=False
)

print(member_type_summary)

In [ ]:
# 7. วิเคราะห์ด้วย SQL
q1 = pd.read_sql_query("""
    SELECT loan_id, book_id, member_id, late_fee
    FROM loans
    WHERE late_fee > 0
    ORDER BY late_fee DESC
""", conn)

print(q1)

In [ ]:
q2 = pd.read_sql_query("""
    SELECT
        l.loan_id,
        b.title,
        b.category_name,
        l.borrow_date
    FROM loans l
    JOIN books b
        ON l.book_id = b.book_id
    WHERE b.category_code = 'D000'
    ORDER BY l.borrow_date DESC
""", conn)

print(q2)

In [ ]:
q3 = pd.read_sql_query("""
    SELECT
        m.name,
        m.member_type,
        l.late_fee
    FROM loans l
    JOIN members m
        ON l.member_id = m.member_id
    WHERE m.member_type = 'อาจารย์'
      AND l.late_fee > 0
    ORDER BY l.late_fee DESC
""", conn)

print(q3)


In [ ]:
q4 = pd.read_sql_query("""
    SELECT
        b.category_name,
        COUNT(*) AS total_loans,
        SUM(l.late_fee) AS total_late_fee
    FROM loans l
    JOIN books b
        ON l.book_id = b.book_id
    GROUP BY b.category_name
    ORDER BY total_late_fee DESC
""", conn)

print(q4)


In [ ]:
top_members = df_merged.groupby(
    "name"
).agg(
    total_borrows=("loan_id", "count")
).reset_index().sort_values(
    "total_borrows",
    ascending=False
).head(5)

print(top_members)

In [ ]:
q5 = pd.read_sql_query("""
    SELECT
        loan_id,
        book_id,
        member_id,
        borrow_date,
        due_date
    FROM loans
    WHERE return_date IS NULL
    ORDER BY due_date ASC
""", conn)

print(q5)

# ขั้นที่ 8 — สร้างกราฟและสรุปผล

In [ ]:
#สร้างกราฟ 3 รูป
import matplotlib.pyplot as plt
#กราฟที่ 1 — จำนวนการยืมตามหมวดหมู่
plt.figure(figsize=(9, 5))

plt.bar(
    category_summary["category_name"],
    category_summary["total_loans"]
)

plt.title("จำนวนการยืมหนังสือแยกตามหมวดหมู่")
plt.xlabel("หมวดหมู่")
plt.ylabel("จำนวนครั้งที่ยืม")

plt.xticks(
    rotation=75,
    ha="right"
)

plt.tight_layout()
plt.show()

#กราฟที่ 2 — ค่าปรับเฉลี่ยตามประเภทสมาชิก
plt.figure(figsize=(6, 5))

plt.bar(
    member_type_summary["member_type"],
    member_type_summary["avg_late_fee"]
)

plt.title("ค่าปรับเฉลี่ยต่อการยืม: นักศึกษา vs อาจารย์")
plt.xlabel("ประเภทสมาชิก")
plt.ylabel("ค่าปรับเฉลี่ย (บาท)")

plt.tight_layout()
plt.show()

#กราฟที่ 3 — Top 5 สมาชิกที่ยืมมากที่สุด
plt.figure(figsize=(8, 5))

plt.barh(
    top_members["name"],
    top_members["total_borrows"]
)

plt.title("5 อันดับสมาชิกที่ยืมหนังสือมากที่สุด")
plt.xlabel("จำนวนครั้งที่ยืม")
plt.ylabel("ชื่อสมาชิก")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()